In [1]:
import pandas as pd
import numpy as np
import gc
import os
import json


def process_vintage_whole_life(file_path, sampling_rate=0.05, random_state=42):
    """
    Process a single vintage file into a quarterly hazard panel.

    Returns
    -------
    df_final : pd.DataFrame
        Hazard panel for this vintage.
    pop_nonevent_rows : int
        Total non-event rows in the full population (before sampling).
        Used to compute tau_eff across all vintages.
    """

    # ── 1. Load with downcasting ──────────────────────────────────────
    df = pd.read_csv(
        file_path, sep="|", header=None,
        usecols=[1, 2, 11, 15, 17, 19, 22, 23, 39],
        dtype={1: "int64", 39: str, 11: "float32",
               19: "float32", 22: "float32", 23: "float32"}
    ).rename(columns={
        1: "ID", 2: "Date", 11: "UPB", 15: "Age", 17: "Rem",
        19: "LTV",  22: "DTI", 23: "FICO", 39: "Delinq"
    })

    for col in df.select_dtypes(include=["int", "int64", "int32"]).columns:
        df[col] = pd.to_numeric(df[col], downcast="integer")
    for col in df.select_dtypes(include=["float", "float64"]).columns:
        df[col] = pd.to_numeric(df[col], downcast="float")

    # ── 2. Timing ─────────────────────────────────────────────────────
    df["t"]      = pd.to_datetime(df["Date"].astype(str).str.zfill(6), format="%m%Y")
    df["Delinq"] = pd.to_numeric(df["Delinq"], errors="coerce").fillna(0)
    df["q"]      = df["t"].dt.to_period("Q").dt.to_timestamp()

    # ── 3. IDs that EVER default — use a set for O(1) lookup ──────────
    default_id_set = set(df.loc[df["Delinq"] >= 3, "ID"])

    # ── 4. Quarterly default flag (monthly → quarterly max) ───────────
    q_default = (
        df.assign(default_in_q=(df["Delinq"] >= 3).astype("int8"))
          .groupby(["ID", "q"], as_index=False)["default_in_q"]
          .max()
    )

    # ── 5. Quarter-end covariate snapshot ────────────────────────────
    # inplace sort avoids creating a temporary copy of the full dataframe
    df.sort_values(["ID", "t"], inplace=True)
    df_q = df.groupby(["ID", "q"]).tail(1).copy()
    df_q = df_q.merge(q_default, on=["ID", "q"], how="left")
    df_q["default_in_q"] = df_q["default_in_q"].fillna(0).astype("int8")
    del df, q_default
    gc.collect()

    # ── 6. ID-based stratified sampling ──────────────────────────────
    is_default_loan = df_q["ID"].isin(default_id_set)
    defaults_history = df_q[is_default_loan].copy()

    healthy_pool = df_q[~is_default_loan]
    healthy_ids  = (
        pd.Series(healthy_pool["ID"].unique())
          .sample(frac=sampling_rate, random_state=random_state)
    )
    # Convert to set for fast lookup
    healthy_id_set  = set(healthy_ids)
    healthy_history = healthy_pool[healthy_pool["ID"].isin(healthy_id_set)].copy()
    del healthy_pool
    gc.collect()

    # ── 7. Hazard panel construction ──────────────────────────────────
    combined = pd.concat([defaults_history, healthy_history], ignore_index=True)
    del defaults_history, healthy_history
    gc.collect()

    # first_default_q: derived from full df_q for correctness,
    # but only defaulted loans have a first_default_q so the source is the same
    first_def = (
        df_q[df_q["default_in_q"] == 1]
        .groupby("ID")["q"]
        .min()
        .rename("first_default_q")
    )
    combined = combined.merge(first_def, on="ID", how="left")

    at_risk  = combined["first_default_q"].isna() | \
               (combined["q"] < combined["first_default_q"])
    df_final = combined[at_risk].copy()
    del combined
    gc.collect()

    # Vectorised quarter shift — much faster than DateOffset row-by-row
    df_final["q_next"] = df_final["q"].dt.to_period("Q").add(1).dt.to_timestamp()
    df_final["target"] = np.int8(0)
    df_final.loc[
        df_final["q_next"] == df_final["first_default_q"], "target"
    ] = np.int8(1)

    # ── 8. Population non-event row count (before sampling) ──────────
    # Merge first_default_q onto the FULL df_q population
    df_q_pop = df_q.merge(first_def, on="ID", how="left")

    # At-risk rows in the population (same filter as hazard panel)
    at_risk_pop = df_q_pop["first_default_q"].isna() | \
                  (df_q_pop["q"] < df_q_pop["first_default_q"])
    total_at_risk_pop = int(at_risk_pop.sum())

    # Exactly one event row per defaulted loan
    n_default_events_pop = len(default_id_set)

    # Non-event rows = at-risk rows minus event rows
    pop_nonevent_rows = total_at_risk_pop - n_default_events_pop

    del df_q, df_q_pop, first_def
    gc.collect()

    return df_final, pop_nonevent_rows


# ── Main loop ─────────────────────────────────────────────────────────

# Auto-generate all quarter files from 2000Q1 to 2011Q4
quarter_files = [
    f"{year}Q{q}.csv"
    for year in range(2000, 2012)
    for q in range(1, 5)
]

all_vintages          = []
total_pop_nonevent    = 0
total_sample_nonevent = 0
chunk_paths           = []

for i, file in enumerate(quarter_files):
    print(f"[{i+1}/{len(quarter_files)}] Processing: {file} ...")

    # Derive a file-specific seed so sampling is reproducible
    # but not identical across files
    file_seed = hash(file) % (2**31)

    chunk, pop_ne = process_vintage_whole_life(
        file, sampling_rate=0.05, random_state=file_seed
    )

    total_pop_nonevent    += pop_ne
    total_sample_nonevent += int((chunk["target"] == 0).sum())

    # Write chunk to disk immediately to avoid holding all chunks in RAM
    chunk_path = f"chunk_{i:03d}.parquet"
    chunk.to_parquet(chunk_path, index=False)
    chunk_paths.append(chunk_path)
    del chunk
    gc.collect()

    print(f"    pop_ne={pop_ne:,}   running_pop_total={total_pop_nonevent:,}")

# ── Compute τ_eff ─────────────────────────────────────────────────────
tau_eff              = total_sample_nonevent / total_pop_nonevent
intercept_correction = float(np.log(tau_eff))

print(f"\n── Sampling Summary ──────────────────────────────────────")
print(f"Population non-event rows : {total_pop_nonevent:,}")
print(f"Sample non-event rows     : {total_sample_nonevent:,}")
print(f"τ_eff                     : {tau_eff:.6f}")
print(f"Intercept correction      : {intercept_correction:.6f}  (add to fitted intercept)")
print(f"Nominal sampling rate     : 0.05")
print(f"Δ (τ_eff − 0.05)         : {tau_eff - 0.05:+.6f}  (from default-loan non-event rows)")

# ── Save τ_eff to JSON (parquet does not persist df.attrs) ────────────
sampling_stats = {
    "tau_eff": tau_eff,
    "intercept_correction": intercept_correction,
    "total_pop_nonevent": total_pop_nonevent,
    "total_sample_nonevent": total_sample_nonevent,
    "sampling_rate": 0.05,
    "files_processed": len(quarter_files)
}
with open("sampling_stats.json", "w") as f:
    json.dump(sampling_stats, f, indent=2)
print("\nSaved sampling_stats.json")

# ── Combine chunks and save ───────────────────────────────────────────
print("\nCombining chunks ...")
df_final = pd.concat(
    [pd.read_parquet(p) for p in chunk_paths],
    ignore_index=True
)

# Clean up temporary chunk files
for p in chunk_paths:
    os.remove(p)

df_final.to_parquet("hazard_panel_full.parquet", index=False)
print(f"Saved hazard_panel_full.parquet  ({len(df_final):,} rows)")


# ── Intercept correction usage (reference) ────────────────────────────
#
# Load stats at modelling time:
#   with open("sampling_stats.json") as f:
#       stats = json.load(f)
#   intercept_correction = stats["intercept_correction"]
#
# After fitting:
#   result = model.fit(cov_type="cluster", cov_kwds={"groups": df["ID"]})
#   corrected_intercept = result.params["const"] + intercept_correction
#
# When predicting:
#   def predict_corrected(result, X, intercept_correction):
#       log_odds = result.predict(X, linear=True) + intercept_correction
#       return 1 / (1 + np.exp(-log_odds))


[1/48] Processing: 2000Q1.csv ...
    pop_ne=3,076,940   running_pop_total=3,076,940
[2/48] Processing: 2000Q2.csv ...
    pop_ne=2,815,486   running_pop_total=5,892,426
[3/48] Processing: 2000Q3.csv ...
    pop_ne=3,017,260   running_pop_total=8,909,686
[4/48] Processing: 2000Q4.csv ...
    pop_ne=3,493,497   running_pop_total=12,403,183
[5/48] Processing: 2001Q1.csv ...
    pop_ne=5,344,857   running_pop_total=17,748,040
[6/48] Processing: 2001Q2.csv ...
    pop_ne=10,834,314   running_pop_total=28,582,354
[7/48] Processing: 2001Q3.csv ...
    pop_ne=9,598,932   running_pop_total=38,181,286
[8/48] Processing: 2001Q4.csv ...
    pop_ne=12,814,481   running_pop_total=50,995,767
[9/48] Processing: 2002Q1.csv ...
    pop_ne=14,274,506   running_pop_total=65,270,273
[10/48] Processing: 2002Q2.csv ...
    pop_ne=8,994,941   running_pop_total=74,265,214
[11/48] Processing: 2002Q3.csv ...
    pop_ne=11,230,041   running_pop_total=85,495,255
[12/48] Processing: 2002Q4.csv ...
    pop_ne=24,38